# ToolRL Reproduction Notebook

This notebook provisions a GPU instance on Chameleon Cloud, sets up the Docker environment, and runs one training case end-to-end so that you can reproduce a number from the paper.

**Target:** GRPO Cold Start, Qwen2.5-1.5B, API-Bank overall accuracy.
**Paper result:** 63.15%. **Our result:** 58.96%.

**Estimated wall time:** ~3 hours on 4x H100 or 4x A100 (15 epochs, ~512 training prompts per epoch).

**Requirements before running:**
- A Chameleon Cloud account with an active allocation
- An SSH keypair registered at chi.uc.chameleoncloud.org
- `python-chi` installed in this notebook's environment: `pip install python-chi`
- Your Chameleon credentials set as environment variables or a `clouds.yaml` file

## 0. Configuration

Set these before running anything else.

In [ ]:
# Your Chameleon site and project
SITE = "CHI@UC"          # or CHI@TACC
PROJECT_NAME = "CHI-XXXXXX"  # replace with your allocation number

# SSH key registered on Chameleon
KEYPAIR_NAME = "my-chameleon-key"          # name as it appears in the dashboard
SSH_KEY_PATH = "~/.ssh/id_rsa"             # local path to the private key

# Node type — must have at least 4 GPUs
NODE_TYPE = "gpu_rtx_6000"   # KVM@CHI@UC H100 node type; change if using a different site

# How long to reserve the instance (hours)
LEASE_HOURS = 6

# Repo to clone on the instance
REPO_URL = "https://github.com/Mario928/toolrl-verl-reproduction"

## 1. Provision the Instance

This creates a lease (reservation), launches a KVM instance under it, and waits for it to be active.

In [ ]:
import chi
import chi.lease
import chi.server
import datetime

chi.use_site(SITE)
chi.set("project_name", PROJECT_NAME)

lease_name = "toolrl-reproduce"
start = datetime.datetime.utcnow() + datetime.timedelta(minutes=1)
end = start + datetime.timedelta(hours=LEASE_HOURS)

lease = chi.lease.create_lease(
    lease_name,
    reservations=[
        chi.lease.get_node_reservation(
            min_count=1,
            max_count=1,
            node_type=NODE_TYPE,
        )
    ],
    start_date=start,
    end_date=end,
)

print(f"Lease created: {lease['name']} (id={lease['id']})")
print(f"Status: {lease['status']} — waiting for ACTIVE...")

chi.lease.wait_for_active(lease["id"])
print("Lease is ACTIVE.")

In [ ]:
reservation_id = chi.lease.get_node_reservation(lease["id"])

server = chi.server.create_server(
    "toolrl-node",
    reservation_id=reservation_id,
    image_name="CC-Ubuntu22.04",
    key_name=KEYPAIR_NAME,
)

chi.server.wait_for_active(server.id)
print(f"Server active: {server.id}")

# Associate a floating IP so we can SSH in
floating_ip = chi.server.associate_floating_ip(server.id)
print(f"Floating IP: {floating_ip}")

## 2. Connect via SSH

Once the floating IP is assigned, open a `RemoteExecutor` that wraps SSH calls.
All subsequent `node.execute(...)` calls run on the remote machine.

In [ ]:
import chi.ssh
import os

SSH_KEY_PATH = os.path.expanduser(SSH_KEY_PATH)

node = chi.ssh.Remote(floating_ip, username="cc", key_filename=SSH_KEY_PATH)

# Sanity check
stdout, stderr = node.execute("uname -a")
print(stdout)

## 3. Install Docker and nvidia-container-toolkit

In [ ]:
node.execute("sudo apt-get update -q")
node.execute(
    "sudo apt-get install -y -q "
    "ca-certificates curl gnupg lsb-release"
)

# Docker repo
node.execute(
    "curl -fsSL https://download.docker.com/linux/ubuntu/gpg "
    "| sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg"
)
node.execute(
    'echo "deb [arch=$(dpkg --print-architecture) '
    'signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] '
    'https://download.docker.com/linux/ubuntu '
    '$(lsb_release -cs) stable" '
    '| sudo tee /etc/apt/sources.list.d/docker.list > /dev/null'
)
node.execute("sudo apt-get update -q")
node.execute(
    "sudo apt-get install -y -q "
    "docker-ce docker-ce-cli containerd.io docker-compose-plugin"
)
node.execute("sudo usermod -aG docker cc")
print("Docker installed.")

In [ ]:
# nvidia-container-toolkit
node.execute(
    "curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey "
    "| sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg"
)
node.execute(
    "curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list "
    "| sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' "
    "| sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list"
)
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q nvidia-container-toolkit")
node.execute("sudo nvidia-ctk runtime configure --runtime=docker")
node.execute("sudo systemctl restart docker")
print("nvidia-container-toolkit installed.")

In [ ]:
# Verify GPUs are visible
stdout, _ = node.execute("sudo docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(stdout)

## 4. Clone the Repo and Create Docker Volumes

In [ ]:
node.execute(f"git clone {REPO_URL} /home/cc/toolrl")

# Create the named volumes that docker-compose.yml references
for vol in ["math-reasoning-rl_models", "math-reasoning-rl_hf_cache",
            "math-reasoning-rl_mlflow_data", "math-reasoning-rl_datasets"]:
    node.execute(f"sudo docker volume create {vol}")

print("Repo cloned and volumes created.")

## 5. Build the Docker Image

The Dockerfile installs torch 2.4.0 + vllm 0.6.3 + flash-attn + the local verl package.
Build takes about 15-20 minutes the first time.

In [ ]:
stdout, stderr = node.execute(
    "cd /home/cc/toolrl && sudo docker build -t toolrl-verl:latest . 2>&1 | tail -20"
)
print(stdout)
if stderr:
    print("STDERR:", stderr)

## 6. Start Containers

In [ ]:
stdout, _ = node.execute(
    "cd /home/cc/toolrl && sudo docker compose up -d"
)
print(stdout)

# Confirm both containers are running
stdout, _ = node.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(stdout)

## 7. Download the Base Model

GRPO cold start uses `Qwen/Qwen2.5-1.5B-Instruct` directly (no SFT checkpoint needed).
We let HuggingFace cache it into the `hf_cache` volume.

In [ ]:
stdout, _ = node.execute(
    "sudo docker exec verl python3 -c \""
    "from huggingface_hub import snapshot_download; "
    "snapshot_download('Qwen/Qwen2.5-1.5B-Instruct')"
    "\""
)
print(stdout if stdout else "Download complete.")

## 8. Run GRPO Cold Start Training

This is the experiment that produces **API-Bank overall accuracy ~59%** (paper: 63.15%).

Key settings:
- `BASE_MODEL` = `Qwen/Qwen2.5-1.5B-Instruct` (HF hub, pulled from cache)
- `COARSEREWARD=0`, `STRICTMATCH=0` (fine-grained reward, paper default)
- 15 epochs, batch size 512, 4 GPUs
- Checkpoint saved every 15 steps; final checkpoint at `global_step_90`

Wall time: ~3 hours on 4x H100.

In [ ]:
train_cmd = """
sudo docker exec -d verl bash -c '
    export CUDA_VISIBLE_DEVICES=0,1,2,3
    export N_GPUS=4
    export ROLLOUT_TP_SIZE=1
    export VLLM_ATTENTION_BACKEND=XFORMERS
    export WITHLENGTH=0
    export REFINEDREWARD=0
    export COARSEREWARD=0
    export STRICTMATCH=0
    export CORRECTMAX1=0
    export MAX1STEP30MAX3=0
    export SCHEDULEREWARD=0
    export SCHEDULELENGTH=0
    export DATA_DIR="./dataset/rlla_4k"
    export BASE_MODEL="Qwen/Qwen2.5-1.5B-Instruct"
    export EXPERIMENT_NAME="/app/models/toolrl-grpo-cold-qwen-1.5b"
    cd /workspace && bash ./examples/grpo_trainer/run_grpo.sh > /tmp/train_grpo_cold.log 2>&1
'
"""

node.execute(train_cmd)
print("Training launched in background. Tail the log with the next cell.")

In [ ]:
# Check training log (run this cell periodically)
stdout, _ = node.execute("sudo docker exec verl tail -30 /tmp/train_grpo_cold.log 2>/dev/null || echo 'log not yet created'")
print(stdout)

In [ ]:
# Check GPU utilization
stdout, _ = node.execute(
    "sudo docker exec verl nvidia-smi --query-gpu=index,memory.used,utilization.gpu "
    "--format=csv,noheader"
)
print(stdout)

## 9. Evaluate: API-Bank

Run this after training finishes (final checkpoint is `global_step_90`).

In [ ]:
CHECKPOINT = "/app/models/toolrl-grpo-cold-qwen-1.5b/actor/global_step_90"

# Generate predictions
gen_cmd = f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 generate.py --model_path {CHECKPOINT} --gpus 0,1,2,3
    > /tmp/apibank_gen.log 2>&1
'
"""
node.execute(gen_cmd)
print("Generation done.")

In [ ]:
# Score predictions
score_cmd = f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_path {CHECKPOINT}
'
"""
stdout, _ = node.execute(score_cmd)
print(stdout)

In [ ]:
# If the generate.py parser bug triggers (raw 3B case only, not 1.5B cold start),
# run the reparse fix instead:
#
# node.execute(
#     f"sudo docker exec verl bash -c 'cd /workspace/benchmarks/API-Bank && "
#     f"python3 reparse_result_fix.py --model_path {CHECKPOINT}'"
# )

## 10. Evaluate: BFCL

BFCL V4 (Berkeley Function Calling Leaderboard). Uses the paper's original handler (`rlla_qwen_original.py`).

In [ ]:
MODEL_NAME = "toolrl-grpo-cold-qwen-1.5b"
CATEGORIES = (
    "simple_python,simple_java,simple_javascript,"
    "multiple,parallel,parallel_multiple,irrelevance,"
    "live_simple,live_multiple,live_parallel,live_parallel_multiple,"
    "live_irrelevance,live_relevance"
)

# Register the model in bfcl model_config.py (one-time setup)
register_cmd = f"""
sudo docker exec verl python3 -c "
import subprocess, sys
config_path = '/usr/local/lib/python3.10/dist-packages/bfcl_eval/constants/model_config.py'
entry = '''
ModelConfig(
    model='{MODEL_NAME}',
    handler=RLLAHandler,
    is_fc=False,
),
'''
with open(config_path, 'r') as f:
    txt = f.read()
if '{MODEL_NAME}' not in txt:
    idx = txt.find('supported_models = [')
    insert = txt.index('[', idx) + 1
    txt = txt[:insert] + entry + txt[insert:]
    with open(config_path, 'w') as f:
        f.write(txt)
    print('Registered.')
else:
    print('Already registered.')
"
"""
stdout, _ = node.execute(register_cmd)
print(stdout)

In [ ]:
# Restart container to free GPU memory before BFCL vLLM inference
node.execute("sudo docker restart verl")
import time; time.sleep(15)
print("Container restarted.")

In [ ]:
bfcl_gen_cmd = f"""
sudo docker exec -d verl bash -c '
    bfcl generate \\
        --model {MODEL_NAME} \\
        --test-category {CATEGORIES} \\
        --backend vllm \\
        --local-model-path {CHECKPOINT} \\
        --num-gpus 4 \\
        --gpu-memory-utilization 0.85 \\
        --allow-overwrite \\
        > /tmp/bfcl_gen.log 2>&1
'
"""
node.execute(bfcl_gen_cmd)
print("BFCL generation launched. Tail /tmp/bfcl_gen.log to monitor.")

In [ ]:
# Monitor generation (run periodically)
stdout, _ = node.execute("sudo docker exec verl tail -20 /tmp/bfcl_gen.log 2>/dev/null")
print(stdout)

In [ ]:
# After generation finishes, score
bfcl_eval_cmd = f"""
sudo docker exec verl bash -c '
    bfcl evaluate \\
        --model {MODEL_NAME} \\
        --test-category {CATEGORIES}
'
"""
stdout, _ = node.execute(bfcl_eval_cmd)
print(stdout)

## 11. Expected Results

| Metric | Paper | This Reproduction |
|---|---|---|
| API-Bank Overall | 63.15% | ~59% |
| BFCL Non-Live AST | — | ~77% |
| BFCL Live Acc | — | ~65% |
| BFCL V4 Overall | 46.20% | ~18% |

The ~4pt API-Bank gap is training variance (single seed, RL training is stochastic).
BFCL V4 Overall is low because `multi_turn` categories are not run here (they require a live server).

See [benchmarks/BFCL/results/bfcl_overall_2026-05-08.csv](benchmarks/BFCL/results/bfcl_overall_2026-05-08.csv) for full scored results across all 5 evaluated cells.

## 12. Cleanup

Delete the lease when done to stop billing against your allocation.

In [ ]:
# Save any result files you want before running this
chi.lease.delete_lease(lease["id"])
print(f"Lease {lease['name']} deleted.")